# EXIOBASE3 hotspot analysis for Sweden with pymrio

This notebook loads the EXIOBASE3 monetary PxP IOT, calculates the missing MRIO tables with `pymrio`, and produces a Sweden-focused hotspot analysis for:

1. economic importance,
2. greenhouse-gas impacts,
3. material footprint,

for both consumption-based accounting (CBA) and production-based accounting (PBA). It also derives source-country shares for the hotspot sectors.

## Key assumptions used in this notebook

- **Economic importance** is represented by the `impact` extension row **`Value Added`**.
- **GHG impacts** are represented by the `impact` extension row **`global warming (GWP100)`**.
- **Material footprint** is represented from the **`materials`** extension. The notebook first tries to find a single “total used-material” style row. If none is found, it sums eligible material rows after checking that the units are consistent.
- **Source-country shares for PBA sectors are trivial by definition**: for Swedish PBA hotspots, the source country is Sweden (`SE`) because PBA records impacts occurring in Swedish production sectors.
- The country-origin decomposition for **CBA** is implemented in a memory-conscious way from `S`, `L`, and Swedish final demand, rather than by fully diagonalizing a stressor, because that approach becomes very memory-intensive for full EXIOBASE systems.

## Before running

- Adjust the settings in the configuration cell if needed.
- Expect parsing and `calc_all()` on EXIOBASE to take noticeable time and memory.

In [1]:
# Optional: install missing packages from inside the notebook
# %pip install pymrio pandas numpy matplotlib openpyxl

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymrio

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# -----------------------------
# User configuration
# -----------------------------
DATA_PATH = Path(r"C:\EXIOBASE3\IOT_2022_pxp.zip")
TARGET_REGION = "SE"

TOP_N_SECTORS = 15          # ranking tables and charts
TOP_N_SOURCE_SECTORS = 5    # how many hotspot sectors per indicator/account to trace to source countries
TOP_N_SOURCE_COUNTRIES = 10 # number of source countries shown per hotspot sector

# Set this to a row label or list of row labels if you want to override
# the automatic material-footprint row selection.
# Examples:
# MATERIAL_ROW_SELECTION = "Total material extraction"
# MATERIAL_ROW_SELECTION = [("Used", "Biomass"), ("Used", "Metal ores")]
MATERIAL_ROW_SELECTION = None

OUTPUT_DIR = Path.cwd() / "exiobase3_sweden_hotspot_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def row_to_text(row_label):
    """Render row labels consistently for exact matching and reporting."""
    if isinstance(row_label, tuple):
        return " | ".join(str(x) for x in row_label)
    return str(row_label)

def as_listlike(row_selector):
    if isinstance(row_selector, (list, tuple, pd.Index, np.ndarray)):
        # tuple may also be a single MultiIndex label, so handle carefully later
        return row_selector
    return [row_selector]

def select_rows(df, row_selector):
    """
    Safe row selection for regular and MultiIndex row labels.
    Returns a DataFrame even for a single row.
    """
    if isinstance(row_selector, tuple) and not isinstance(df.index, pd.MultiIndex):
        return df.loc[[row_selector]]
    if isinstance(row_selector, (list, pd.Index, np.ndarray)):
        return df.loc[list(row_selector)]
    return df.loc[[row_selector]]

def sum_selected_rows(df, row_selector):
    selected = select_rows(df, row_selector)
    if selected.shape[0] == 1:
        return selected.iloc[0]
    return selected.sum(axis=0)

def aggregate_final_demand_by_region(Y):
    """
    Aggregate final demand categories to one column per region.
    """
    if isinstance(Y.columns, pd.MultiIndex):
        col_level = "region" if "region" in Y.columns.names else 0
        return Y.T.groupby(level=col_level, sort=False).sum().T
    return Y.copy()

def split_region_sector_columns(df, target_region):
    """
    Return the sector columns for a specific region from a standard pymrio account table.
    """
    if isinstance(df.columns, pd.MultiIndex):
        level = "region" if "region" in df.columns.names else 0
        return df.xs(target_region, axis=1, level=level)
    raise ValueError("Expected a pymrio-style DataFrame with MultiIndex columns (region, sector).")

def ranking_from_account(account_df, row_selector, target_region, account_name, indicator_name, unit=None):
    """
    Build a tidy ranking table for one indicator and one account type.
    """
    regional_df = split_region_sector_columns(account_df, target_region)
    values = sum_selected_rows(regional_df, row_selector).astype(float)
    values = values.sort_values(ascending=False)

    total = values.sum()
    out = (
        values.rename("value")
        .reset_index()
        .rename(columns={"index": "sector", "sector": "sector"})
    )
    if "sector" not in out.columns:
        out.columns = ["sector", "value"]

    out["account"] = account_name
    out["indicator"] = indicator_name
    out["region"] = target_region
    out["share_of_region_total"] = np.where(total != 0, out["value"] / total, np.nan)
    out["rank"] = np.arange(1, len(out) + 1)
    out["unit"] = unit
    return out[["account", "indicator", "region", "sector", "value", "share_of_region_total", "rank", "unit"]]

def detect_common_unit(ext, row_selector):
    """
    Try to detect a common unit across selected extension rows.
    """
    unit_obj = getattr(ext, "unit", None)
    if unit_obj is None:
        return None

    try:
        if isinstance(unit_obj, pd.DataFrame):
            selected = select_rows(unit_obj, row_selector)
            if selected.shape[1] == 1:
                units = selected.iloc[:, 0].astype(str).str.strip().dropna().unique().tolist()
            else:
                units = pd.unique(selected.astype(str).stack().str.strip()).tolist()
        elif isinstance(unit_obj, pd.Series):
            selected = unit_obj.loc[list(select_rows(ext.F, row_selector).index)]
            units = selected.astype(str).str.strip().dropna().unique().tolist()
        else:
            return None

        units = [u for u in units if u and u.lower() != "nan"]
        if len(units) == 1:
            return units[0]
        if len(units) == 0:
            return None
        raise ValueError(f"Selected rows do not have a single common unit: {units}")
    except Exception:
        return None

def choose_material_rows(materials_ext, override=None):
    """
    Heuristic material-row selector.

    Strategy:
    1. Use explicit override if provided.
    2. Prefer a single row that looks like a total used-material row.
    3. Otherwise sum rows that look like 'used' material rows and exclude 'unused'.
    4. As last resort, sum all rows only if units are consistent.
    """
    if override is not None:
        return override, "User-specified material row selection"

    rows = list(materials_ext.get_rows())
    row_text = pd.Series({row: row_to_text(row).lower() for row in rows})

    # Preferred exact-ish patterns
    preferred_rows = []
    for row, txt in row_text.items():
        if ("total" in txt or "all" in txt) and ("material" in txt or "domestic extraction" in txt or "used" in txt) and "unused" not in txt:
            preferred_rows.append(row)

    if len(preferred_rows) == 1:
        return preferred_rows[0], f"Automatically selected material row: {row_to_text(preferred_rows[0])}"

    used_rows = []
    for row, txt in row_text.items():
        include = (
            ("material" in txt or "biomass" in txt or "metal" in txt or "non-metal" in txt or "fossil" in txt or "ore" in txt)
            and "unused" not in txt
        )
        if include:
            used_rows.append(row)

    if used_rows:
        unit = detect_common_unit(materials_ext, used_rows)
        if unit is not None:
            return used_rows, f"Automatically summed {len(used_rows)} material rows with common unit: {unit}"

    all_unit = detect_common_unit(materials_ext, rows)
    if all_unit is not None:
        return rows, f"No clear total material row found; summed all material rows with common unit: {all_unit}"

    raise ValueError(
        "Could not determine a safe material-footprint row selection automatically. "
        "Inspect materials rows below and set MATERIAL_ROW_SELECTION manually."
    )

def cba_source_country_shares(mrio, ext, row_selector, target_region, target_sector, Y_agg, top_n=10):
    """
    Source-country decomposition for one Sweden sector hotspot under CBA.

    This avoids creating a full diagonalized stressor extension.
    From pymrio's account formulation:
        Y_diag = diagonalize_columns_to_sectors(Y_agg)
        x_diag = L @ Y_diag
        D_cba = S @ x_diag

    For one target column (region, sector), the source-sector contributions are:
        s_i * L[i, j] * y_j
    where y_j is the final-demand scalar for that same region-sector entry.
    """
    target_col = (target_region, target_sector)

    S_row = sum_selected_rows(ext.S, row_selector).astype(float)
    L_col = mrio.L.loc[:, target_col].astype(float)
    y_scalar = float(Y_agg.loc[target_col, target_region])

    source_sector_values = S_row * L_col * y_scalar
    source_country_values = (
        source_sector_values.groupby(level="region", sort=False).sum()
        .sort_values(ascending=False)
    )

    total = source_country_values.sum()
    out = source_country_values.rename("source_value").reset_index()
    out.columns = ["source_country", "source_value"]
    out["source_share"] = np.where(total != 0, out["source_value"] / total, np.nan)
    out["target_region"] = target_region
    out["target_sector"] = target_sector
    out["rank_source_country"] = np.arange(1, len(out) + 1)
    return out.head(top_n)

def pba_source_country_shares(target_region, target_sector, value):
    """
    PBA country source is trivial by definition for domestic sectors.
    """
    return pd.DataFrame(
        {
            "source_country": [target_region],
            "source_value": [float(value)],
            "source_share": [1.0],
            "target_region": [target_region],
            "target_sector": [target_sector],
            "rank_source_country": [1],
        }
    )

def plot_top_sectors(ranking_df, title, top_n=15):
    top = ranking_df.head(top_n).iloc[::-1]
    plt.figure(figsize=(10, max(6, 0.35 * len(top))))
    plt.barh(top["sector"], top["value"])
    plt.xlabel(top["unit"].dropna().iloc[0] if ranking_df["unit"].notna().any() else "Value")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [4]:
# Check that the file exists before starting heavy calculations
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find EXIOBASE at: {DATA_PATH}\n"
        "Update DATA_PATH in the configuration cell to match your local file."
    )

print(f"Loading EXIOBASE from: {DATA_PATH}")
mrio = pymrio.parse_exiobase3(DATA_PATH)

print("Calculating missing MRIO tables and accounts. This can take substantial time and memory.")
mrio.calc_all()

print("Loaded regions:", len(mrio.get_regions()))
print("Loaded sectors:", len(mrio.get_sectors()))
print("Extensions:", list(mrio.get_extensions()))

Loading EXIOBASE from: C:\EXIOBASE3\IOT_2022_pxp.zip
Calculating missing MRIO tables and accounts. This can take substantial time and memory.
Loaded regions: 49
Loaded sectors: 200
Extensions: ['impacts', 'satellite']


In [5]:
# Basic checks and quick inspection
assert TARGET_REGION in mrio.get_regions(), f"{TARGET_REGION} is not a valid EXIOBASE region code."

print("Impact rows (first 30):")
display(pd.DataFrame({"impact_row": list(mrio.impact.get_rows())[:30]}))

print("Material rows (first 50):")
display(pd.DataFrame({"material_row": [row_to_text(x) for x in list(mrio.materials.get_rows())[:50]]}))

Impact rows (first 30):


AttributeError: 'IOSystem' object has no attribute 'impact'

In [ ]:
# Prepare the final-demand aggregation used by pymrio account calculations
Y_agg = aggregate_final_demand_by_region(mrio.Y)

print("Y aggregated to one column per region:")
display(Y_agg.iloc[:5, :5])

print("Selected region:", TARGET_REGION)

In [ ]:
# Indicator definitions
value_added_row = "Value Added"
ghg_row = "global warming (GWP100)"

material_rows, material_note = choose_material_rows(mrio.materials, override=MATERIAL_ROW_SELECTION)

indicator_specs = {
    "economic_importance": {
        "extension": mrio.impact,
        "row_selector": value_added_row,
        "unit": detect_common_unit(mrio.impact, value_added_row),
        "label": "Value added",
    },
    "ghg": {
        "extension": mrio.impact,
        "row_selector": ghg_row,
        "unit": detect_common_unit(mrio.impact, ghg_row),
        "label": "Global warming (GWP100)",
    },
    "materials": {
        "extension": mrio.materials,
        "row_selector": material_rows,
        "unit": detect_common_unit(mrio.materials, material_rows),
        "label": "Material footprint",
    },
}

print(material_note)
print("Material row selection:")
if isinstance(material_rows, (list, pd.Index, np.ndarray)):
    display(pd.DataFrame({"selected_material_rows": [row_to_text(x) for x in material_rows]}))
else:
    display(pd.DataFrame({"selected_material_rows": [row_to_text(material_rows)]}))

In [ ]:
# Build Sweden hotspot rankings for CBA and PBA
ranking_tables = []

for indicator_name, spec in indicator_specs.items():
    ext = spec["extension"]
    row_selector = spec["row_selector"]
    unit = spec["unit"]

    ranking_tables.append(
        ranking_from_account(
            ext.D_cba,
            row_selector=row_selector,
            target_region=TARGET_REGION,
            account_name="CBA",
            indicator_name=indicator_name,
            unit=unit,
        )
    )

    ranking_tables.append(
        ranking_from_account(
            ext.D_pba,
            row_selector=row_selector,
            target_region=TARGET_REGION,
            account_name="PBA",
            indicator_name=indicator_name,
            unit=unit,
        )
    )

rankings = pd.concat(ranking_tables, ignore_index=True)
rankings.to_csv(OUTPUT_DIR / "sweden_sector_rankings_tidy.csv", index=False)

display(rankings.head(20))
print(f"Saved: {OUTPUT_DIR / 'sweden_sector_rankings_tidy.csv'}")

In [ ]:
# Display top sectors by indicator and accounting principle
for account_name in ["CBA", "PBA"]:
    for indicator_name, spec in indicator_specs.items():
        title = f"{account_name} hotspots for Sweden – {spec['label']}"
        subset = (
            rankings
            .query("account == @account_name and indicator == @indicator_name")
            .sort_values("rank")
            .head(TOP_N_SECTORS)
            .reset_index(drop=True)
        )
        display(subset)
        plot_top_sectors(subset, title=title, top_n=TOP_N_SECTORS)

In [ ]:
# Build source-country shares for top hotspot sectors
source_tables = []

for account_name in ["CBA", "PBA"]:
    for indicator_name, spec in indicator_specs.items():
        ext = spec["extension"]
        row_selector = spec["row_selector"]
        unit = spec["unit"]

        hotspot_subset = (
            rankings
            .query("account == @account_name and indicator == @indicator_name")
            .sort_values("rank")
            .head(TOP_N_SOURCE_SECTORS)
            .copy()
        )

        for _, row in hotspot_subset.iterrows():
            sector = row["sector"]
            sector_value = row["value"]

            if account_name == "CBA":
                src = cba_source_country_shares(
                    mrio=mrio,
                    ext=ext,
                    row_selector=row_selector,
                    target_region=TARGET_REGION,
                    target_sector=sector,
                    Y_agg=Y_agg,
                    top_n=TOP_N_SOURCE_COUNTRIES,
                )
            else:
                src = pba_source_country_shares(
                    target_region=TARGET_REGION,
                    target_sector=sector,
                    value=sector_value,
                )

            src["account"] = account_name
            src["indicator"] = indicator_name
            src["target_sector_value"] = sector_value
            src["unit"] = unit
            source_tables.append(src)

source_country_shares = pd.concat(source_tables, ignore_index=True)
source_country_shares.to_csv(OUTPUT_DIR / "sweden_source_country_shares.csv", index=False)

display(source_country_shares.head(30))
print(f"Saved: {OUTPUT_DIR / 'sweden_source_country_shares.csv'}")

In [ ]:
# Optional: a compact wide-format export for easy browsing in Excel
rankings_wide = (
    rankings
    .pivot_table(
        index=["region", "sector"],
        columns=["account", "indicator"],
        values="value",
        aggfunc="first",
    )
    .sort_index(axis=1)
)

source_shares_wide = (
    source_country_shares
    .pivot_table(
        index=["account", "indicator", "target_region", "target_sector"],
        columns="source_country",
        values="source_share",
        aggfunc="first",
    )
    .sort_index(axis=1)
)

with pd.ExcelWriter(OUTPUT_DIR / "sweden_hotspot_analysis.xlsx", engine="openpyxl") as writer:
    rankings.to_excel(writer, sheet_name="rankings_tidy", index=False)
    rankings_wide.to_excel(writer, sheet_name="rankings_wide")
    source_country_shares.to_excel(writer, sheet_name="source_shares_tidy", index=False)
    source_shares_wide.to_excel(writer, sheet_name="source_shares_wide")

print(f"Saved: {OUTPUT_DIR / 'sweden_hotspot_analysis.xlsx'}")

## Notes for interpretation

- **CBA rankings** show which Swedish final-demand sectors are associated with the largest global value added, GHG burden, or material footprint.
- **PBA rankings** show which Swedish production sectors account for the largest impacts occurring in Sweden.
- For **PBA**, the source-country result is reported as **SE = 100%** because the sector impacts occur in Swedish production.
- For **CBA**, the country-source decomposition is calculated from the same underlying `pymrio` account logic but only for the requested hotspot columns, which keeps memory use manageable.
- If you prefer a different material-footprint definition, set `MATERIAL_ROW_SELECTION` explicitly and rerun the notebook.

## Files written by this notebook

The notebook writes the following files to `OUTPUT_DIR`:

- `sweden_sector_rankings_tidy.csv`
- `sweden_source_country_shares.csv`
- `sweden_hotspot_analysis.xlsx`